# regxp1 — does RFM dimensionality predict steerability?

**Hypothesis.** A concept is steerable when its steering vector has a *dominant top-1 direction*; if not, the concept is encoded as a composite of directions and is harder to steer.

**Data.** Llama-3.1-8B. 90 concepts (30 each: places, personalities, moods). Per-concept per-layer RFM fit stats live in `directions/rfm_<concept>_llama_3_8b_it_eng_only_rfmstats.pkl`, keyed by negative layer index (-1 = last block). Steering judge labels (GPT-OSS) are in `cached_outputs/` for prompt versions v1 (no suffix) and v4 (`_v4`).

**Metrics tested per layer** (from `rfm.py::get_top_dir_err`):
- `rsq` (`val_r`): |Pearson corr| of the RFM probe on held-out validation (the paper's own selection metric).
- `frac1` = lambda1 / trace : share of the AGOP spectrum captured by the top eigenvector (top-1 dominance).
- `frac2` = lambda1 / lambda2 : ratio of the two largest AGOP eigenvalues (lobpcg k=2).

**DV.** `y = 1` if the concept was steered on **either** prompt v1 **or** v4, `y = 0` only if both failed. (No class adjustment — under the hypothesis, class differences in steerability flow *through* dimensionality, so controlling for class would be a bad control.)

In [1]:
import pickle, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import statsmodels.api as sm
np.random.seed(0)

## 1. Build the dataset (one row per concept, N = 90)

In [2]:
classes = ['places', 'personalities', 'moods']
keys = list(range(-1, -32, -1))          # all 31 RFM layers; key -1 = last block
def depth(L): return 32 + L              # depth 1 = first block ... 31 = last

rows = []
for cls in classes:
    jv1 = pickle.load(open(f'cached_outputs/rfm_{cls}_gpt_oss_outputs_500_concepts_llama_3.1_8B_english_only.pkl', 'rb'))
    jv4 = pickle.load(open(f'cached_outputs/rfm_{cls}_gpt_oss_outputs_500_concepts_llama_3.1_8B_english_only_v4.pkl', 'rb'))
    for c in jv1:
        st = pickle.load(open(f'directions/rfm_{c}_llama_3_8b_it_eng_only_rfmstats.pkl', 'rb'))
        y = 1 if (jv1[c] == 1 or jv4[c] == 1) else 0      # DV: steered on EITHER v1 or v4
        b = dict(concept=c, cls=cls, y=y)
        for L in keys:
            b[f'rsq_{depth(L)}'] = st[L]['val_r']         # rsq
            b[f'f1_{depth(L)}']  = st[L]['frac1']         # lambda1 / trace
            b[f'f2_{depth(L)}']  = st[L]['frac2']         # lambda1 / lambda2
        rows.append(b)

df = pd.DataFrame(rows)
print('N =', len(df), '| y=1:', int(df.y.sum()), '| y=0:', int((1 - df.y).sum()), '| mean:', round(df.y.mean(), 3))
df[['concept', 'cls', 'y', 'f1_9', 'f1_29']].head()

N = 90 | y=1: 65 | y=0: 25 | mean: 0.722


,concept,cls,y,f1_9,f1_29
0,Lisbon,places,1,0.479203,0.650730
1,Tampa,places,1,0.490843,0.775671
2,Louisville,places,0,0.477631,0.850832
3,Austin,places,0,0.465480,0.780137
4,Edinburgh,places,1,0.507512,0.772741


## 2. The regression

For each layer L and each metric we fit one **univariate logistic regression** (a single predictor, so no multicollinearity):

$$\operatorname{logit} P(y=1) = b_0 + b_1 \cdot \text{metric}_L$$

Each model is scored by **5-fold stratified cross-validated AUC**: split the 90 concepts into 5 folds, train on 4, predict the held-out fold, pool the held-out predictions, compute AUC once. This measures out-of-sample ranking, not in-sample fit.

In [3]:
y = df['y'].values
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

def cv_auc(x):
    """5-fold stratified CV-AUC of a 1-predictor logistic regression."""
    x = x.reshape(-1, 1); oof = np.zeros(len(y))
    for tr, te in skf.split(x, y):
        oof[te] = LogisticRegression(C=1e6, max_iter=2000).fit(x[tr], y[tr]).predict_proba(x[te])[:, 1]
    return roc_auc_score(y, oof)

results = {}
for pref, name in [('rsq', 'rsq (val_r)'), ('f1', 'lambda1/trace'), ('f2', 'lambda1/lambda2')]:
    res = sorted([(depth(L), cv_auc(df[f'{pref}_{depth(L)}'].values)) for L in keys], key=lambda t: -t[1])
    results[pref] = res
    print(f'=== {name}: CV-AUC per layer (top 8 of 31) ===')
    for d, a in res[:8]:
        print(f'  depth {d:>2}: AUC={a:.3f}')
    print(f'  ... worst: depth {res[-1][0]} AUC={res[-1][1]:.3f}\n')

=== rsq (val_r): CV-AUC per layer (top 8 of 31) ===
  depth  4: AUC=0.863
  depth 31: AUC=0.819
  depth 30: AUC=0.796
  depth 29: AUC=0.785
  depth 28: AUC=0.783
  depth 27: AUC=0.774
  depth 26: AUC=0.767
  depth  8: AUC=0.756
  ... worst: depth 12 AUC=0.271



=== lambda1/trace: CV-AUC per layer (top 8 of 31) ===
  depth  9: AUC=0.858
  depth 29: AUC=0.798
  depth 30: AUC=0.788
  depth  2: AUC=0.777
  depth 28: AUC=0.766
  depth 27: AUC=0.766
  depth 20: AUC=0.743
  depth 19: AUC=0.743
  ... worst: depth 3 AUC=0.417

=== lambda1/lambda2: CV-AUC per layer (top 8 of 31) ===
  depth  9: AUC=0.790
  depth 29: AUC=0.780
  depth 28: AUC=0.764
  depth 30: AUC=0.763
  depth 19: AUC=0.762
  depth 27: AUC=0.756
  depth 18: AUC=0.756
  depth 26: AUC=0.751
  ... worst: depth 10 AUC=0.394



## 3. Direction of the effect (sign of b1) by depth

AUC measures separation strength but not direction. Here we read the raw (unstandardized) logistic coefficient `b1` to see whether *higher* dimensionality means *more* or *less* steerable, at four depths.

In [4]:
print('lambda1/trace: sign of effect (raw logit b1) by depth')
for L in [-30, -23, -12, -3]:        # depth 2, 9, 20, 29
    col = f'f1_{depth(L)}'; x = df[col].values
    m = sm.Logit(y, sm.add_constant(x)).fit(disp=0); b1 = m.params[1]
    tag = 'POS supports (concentrated -> steerable)' if b1 > 0 else 'NEG contradicts'
    print(f'  depth {depth(L):>2}: b1={b1:+.2f}  {tag} | steered mean={df[df.y==1][col].mean():.3f} vs not={df[df.y==0][col].mean():.3f}')

lambda1/trace: sign of effect (raw logit b1) by depth
  depth  2: b1=+27.22  POS supports (concentrated -> steerable) | steered mean=0.437 vs not=0.361
  depth  9: b1=+37.69  POS supports (concentrated -> steerable) | steered mean=0.557 vs not=0.482
  depth 20: b1=-11.50  NEG contradicts | steered mean=0.609 vs not=0.694


  depth 29: b1=-11.87  NEG contradicts | steered mean=0.616 vs not=0.753


## 4. Mechanism: mean lambda1/trace by class, early vs late layer

In [5]:
print('Mean lambda1/trace by class (early depth 9 vs late depth 29):')
for cls in classes:
    s = df[df.cls == cls]
    print(f'  {cls:14s} steer-rate={s.y.mean():.2f}  f1@9={s["f1_9"].mean():.3f}  f1@29={s["f1_29"].mean():.3f}')

Mean lambda1/trace by class (early depth 9 vs late depth 29):
  places         steer-rate=0.37  f1@9=0.480  f1@29=0.772
  personalities  steer-rate=0.80  f1@9=0.550  f1@29=0.677
  moods          steer-rate=1.00  f1@9=0.578  f1@29=0.513


## 5. Findings (last run)

- **Best robust predictor: lambda1/trace at depth 9, CV-AUC = 0.858**, positive sign (concentrated -> steerable). `rsq` peaks slightly higher at depth 4 (0.863) but is erratic across depth (down to 0.27, below chance) because it is saturated -- not a stable signal. `lambda1/lambda2` also peaks at depth 9 (0.790).
- **The sign flips with depth.** Early/mid layers (<= ~9): top-1 dominance predicts steerability (supports hypothesis). Late layers (>= ~20): it reverses.
- **Why:** steerable concepts (moods) are concentrated *early* (f1@9 = 0.578) then spread out *late* (f1@29 = 0.513). Unsteerable concepts (places) are the opposite -- diffuse early (f1@9 = 0.480), then collapse to a dominant direction late (f1@29 = 0.772), most likely the representation snapping onto the literal output token (the place name) rather than a steerable concept direction.
- **Takeaway:** the hypothesis holds at the early-to-mid block (~depth 9); late-layer dominance is a *counter*-signal (lexical collapse).

## Note: AUC and CV-AUC

**AUC** = Area Under the ROC Curve. It equals the probability that a randomly chosen positive (steerable) concept is ranked above a randomly chosen negative (not-steerable) concept by the model's predicted probability. 0.5 = chance, 1.0 = perfect separation. It is a *ranking* metric, not accuracy, and is unaffected by class imbalance.

**CV-AUC** = Cross-Validated AUC: the same quantity but computed out-of-sample. The concepts are split into folds; each fold is predicted by a model trained on the others; the held-out predictions are pooled and scored once. This guards against overfitting -- an in-sample AUC can look high simply because the model memorized the training rows; CV-AUC reports how well the relationship generalizes to concepts the model did not see.

## 6. Two-predictor model: `steerability ~ rsq_L + l1_trace_L`

Same per-layer setup, but each logistic model now uses **both** predictors at the same layer:

$$\operatorname{logit} P(y=1) = b_0 + b_1\,\text{rsq}_L + b_2\,\text{l1\_trace}_L$$

Question: does combining them raise CV-AUC, and do the best layers change versus the single-predictor models in section 2?

In [6]:
def cv_auc_multi(X):
    """5-fold stratified CV-AUC for a logistic regression with >=1 predictors."""
    X = np.atleast_2d(X); oof = np.zeros(len(y))
    for tr, te in skf.split(X, y):
        oof[te] = LogisticRegression(C=1e6, max_iter=5000).fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
    return roc_auc_score(y, oof)

res_both = []
for L in keys:
    d = depth(L)
    X = df[[f'rsq_{d}', f'f1_{d}']].values
    res_both.append((d, cv_auc_multi(X)))
res_both = sorted(res_both, key=lambda t: -t[1])

print('=== steerability ~ rsq_L + l1_trace_L (both, per layer): CV-AUC, top 10 of 31 ===')
for d, a in res_both[:10]:
    print(f'  depth {d:>2}: AUC={a:.3f}')
print(f'  ... worst: depth {res_both[-1][0]} AUC={res_both[-1][1]:.3f}')

# side-by-side vs the single-predictor bests from section 2
rsq_best = {d: a for d, a in results['rsq']}
f1_best  = {d: a for d, a in results['f1']}
print('\nlayer | rsq_only | l1trace_only | both')
for d, a in res_both[:6]:
    print(f'  depth {d:>2}: {rsq_best[d]:.3f}      {f1_best[d]:.3f}        {a:.3f}')

=== steerability ~ rsq_L + l1_trace_L (both, per layer): CV-AUC, top 10 of 31 ===
  depth  4: AUC=0.857
  depth  9: AUC=0.855
  depth  2: AUC=0.813
  depth 31: AUC=0.802
  depth 29: AUC=0.802
  depth 30: AUC=0.772
  depth 28: AUC=0.759
  depth 27: AUC=0.753
  depth 26: AUC=0.746
  depth 20: AUC=0.743
  ... worst: depth 1 AUC=0.388

layer | rsq_only | l1trace_only | both
  depth  4: 0.863      0.647        0.857
  depth  9: 0.718      0.858        0.855
  depth  2: 0.755      0.777        0.813
  depth 31: 0.819      0.580        0.802
  depth 29: 0.785      0.798        0.802
  depth 30: 0.796      0.788        0.772


**Result.** Combining `rsq` and `l1_trace` does **not** raise out-of-sample AUC over the best single predictor: depth 4 goes 0.863 (rsq alone) -> 0.857 (both), depth 9 goes 0.858 (l1_trace alone) -> 0.855 (both) -- a tiny *decrease*, the usual CV penalty for an extra parameter that adds no new signal. The leading layers are unchanged (depth 4 and depth 9 still top the list). So at the layers where each metric is informative, `rsq` and `l1_trace` are largely **redundant** with each other; the second predictor is not contributing independent information about steerability.

## 7. Are `rsq` and `l1_trace` measuring the same thing? (No.)

Section 6 showed that adding both predictors does not beat the best single one. That could mean they are redundant, OR that only one carries signal at each layer. This checks directly: the per-layer correlation between `rsq` and `l1_trace`, and how much `rsq` even varies.

In [7]:
from scipy.stats import pearsonr, spearmanr
print('depth | rsq mean | rsq std | corr(rsq, l1_trace)  Pearson / Spearman')
for d in [2, 4, 9, 16, 20, 29, 31]:
    rsq = df[f'rsq_{d}'].values; f1 = df[f'f1_{d}'].values
    print(f'  {d:>2}   {rsq.mean():.3f}    {rsq.std():.3f}    {pearsonr(rsq,f1)[0]:+.2f} / {spearmanr(rsq,f1)[0]:+.2f}')

depth | rsq mean | rsq std | corr(rsq, l1_trace)  Pearson / Spearman
   2   0.918    0.027    +0.30 / +0.42
   4   0.912    0.018    -0.54 / -0.53
   9   0.994    0.005    -0.35 / -0.44
  16   0.999    0.000    +0.17 / +0.19
  20   0.999    0.001    +0.51 / +0.64
  29   0.996    0.015    +0.37 / +0.76
  31   0.981    0.080    +0.05 / +0.40


**They are different constructs.** The correlation is weak and *flips sign by layer* (-0.54 at depth 4, +0.76 Spearman at depth 29) -- not what redundant measurements look like. The reason combining them did not help (Section 6) is instead: (1) `rsq` is **saturated** -- its std collapses to ~0 by depth 16, so it has no variance to contribute at the layers where `l1_trace` works; and (2) where `rsq` does vary (depth 4), `l1_trace` simply is not predictive of steerability there. So at any given layer only one of the two carries usable signal -- not because they overlap, but because of *where* each has variance.

## 8. Why only ~0.86 and not 1.0? Error analysis at depth 9 (`l1_trace`)

Fit the single-predictor logistic at depth 9, threshold at P=0.5, and inspect the misclassified concepts and whether the effect survives *within* a class.

In [8]:
import statsmodels.api as sm
x = df['f1_9'].values
m = sm.Logit(y, sm.add_constant(x)).fit(disp=0); b0, b1 = m.params; thr = -b0/b1
pred = (m.predict(sm.add_constant(x)) > 0.5).astype(int)
print(f'logit P = {b0:.2f} + {b1:.2f} * l1_trace ; P=0.5 at l1_trace={thr:.3f} ; correct={int((pred==y).sum())}/90')
print(f'steerable mean l1@9={x[y==1].mean():.3f}  vs not-steerable={x[y==0].mean():.3f}\n')

fn = df[(y==1) & (pred==0)]   # steerable despite LOW l1_trace
fp = df[(y==0) & (pred==1)]   # NOT steerable despite HIGH l1_trace
print(f'TYPE A  steerable despite LOW l1_trace (n={len(fn)}):')
print(fn[['concept','cls','f1_9']].to_string(index=False))
print(f'\nTYPE B  NOT steerable despite HIGH l1_trace (n={len(fp)}):')
print(fp[['concept','cls','f1_9']].to_string(index=False))

from scipy.stats import pointbiserialr
print('\nWITHIN-class corr(y, l1_trace@9) -- does the effect survive inside a class?')
for cls in ['places','personalities','moods']:
    s = df[df.cls==cls]
    if s.y.nunique() < 2:
        print(f'  {cls:14s}: all steerable (no within-class variation)'); continue
    print(f'  {cls:14s}: corr={pointbiserialr(s.y, s.f1_9)[0]:+.2f}  (steer {s[s.y==1].f1_9.mean():.3f} vs not {s[s.y==0].f1_9.mean():.3f})')
print('\nBETWEEN-class:')
print(df.groupby('cls').agg(steer_rate=('y','mean'), l1_9_mean=('f1_9','mean')).round(3).to_string())

logit P = -18.30 + 37.69 * l1_trace ; P=0.5 at l1_trace=0.486 ; correct=72/90
steerable mean l1@9=0.557  vs not-steerable=0.482

TYPE A  steerable despite LOW l1_trace (n=9):
                     concept           cls     f1_9
                      Lisbon        places 0.479203
                    Honolulu        places 0.471594
                 Minneapolis        places 0.476643
                      Dublin        places 0.483810
                      Berlin        places 0.454797
                     Jakarta        places 0.461968
          algorithm designer personalities 0.479400
              data scientist personalities 0.483884
computational neuroscientist personalities 0.461002

TYPE B  NOT steerable despite HIGH l1_trace (n=9):
                concept           cls     f1_9
           Kuala Lumpur        places 0.504780
              Nashville        places 0.510348
                  Boise        places 0.504098
          Oklahoma City        places 0.505363
            San An

**Findings.**
- Errors are **symmetric**: 9 Type A (steerable despite low `l1_trace`) and 9 Type B (not steerable despite high `l1_trace`), 18 total, all sitting in the narrow overlap band ~0.45-0.54 around the 0.486 threshold. Zero errors in moods (cleanly separated, high `l1_trace`); all errors are in places and personalities.
- The hypothesis direction **holds at every level and never reverses**: between-class is monotone (moods 0.578 / personalities 0.550 / places 0.480, matching steer rates 1.00 / 0.80 / 0.37), and within-class the correlation is positive (personalities +0.50, places +0.19).
- But within places the separation is tiny (steerable 0.485 vs not 0.477), so `l1_trace` cannot pick out *which* place steers -- the bulk of its power is the coarse between-class (concept-like vs lexical) distinction. That overlap is what caps AUC at ~0.86 rather than 1.0.